In [1]:
import pandas as pd

In [14]:
df = pd.read_csv('datos/respondents.csv')
df.head(2)

,ResponseId,LanguageHaveWorkedWith,LanguageWantToWorkWith,LanguageAdmired,DatabaseHaveWorkedWith,DatabaseWantToWorkWith,DatabaseAdmired,PlatformHaveWorkedWith,PlatformWantToWorkWith,PlatformAdmired,NEWCollabToolsHaveWorkedWith,NEWCollabToolsWantToWorkWith,NEWCollabToolsAdmired,TechEndorse,YearsCode,YearsCodePro,Country,EdLevel_Group,Age_Group,RemoteWork_Clean
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USA,No Formal Degree,Under 18,Remote
1,2,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Bash/Shell (all shells);Go;HTML/CSS;Java;JavaS...,Dynamodb;MongoDB;PostgreSQL,PostgreSQL,PostgreSQL,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,Amazon Web Services (AWS);Heroku;Netlify,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm,PyCharm;Visual Studio Code;WebStorm,NaN,20.0,17.0,GBR,Undergraduate Degree,35-44,Remote


In [15]:
df['ResponseId'].nunique()

65437

In [22]:
cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'LanguageAdmired',
    'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'DatabaseAdmired',
    'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'PlatformAdmired',
    'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'NEWCollabToolsAdmired'
]

df_no_response = df[df[cols].isna().all(axis=1)]

df_no_response['ResponseId'].nunique()

5378

In [23]:
65437 - 5378

60059

In [4]:
def explode_multiresponse(df, column, category, relation_name, entity_name, delimiter=';'):
    temp = df[['ResponseId', column]].dropna().copy()
    temp[column] = temp[column].str.split(delimiter)
    temp = temp.explode(column)
    temp[column] = temp[column].str.strip()

    temp['Category'] = category 
    temp = temp.rename(columns={column: entity_name})
    temp['Relation'] = relation_name
    
    return temp

In [5]:
languages = pd.concat([
    explode_multiresponse(df, 'LanguageHaveWorkedWith', 'Language', 'HaveWorkedWith', 'Technology'),
    explode_multiresponse(df, 'LanguageWantToWorkWith', 'Language', 'WantToWorkWith', 'Technology'),
    explode_multiresponse(df, 'LanguageAdmired', 'Language', 'Admired', 'Technology')
], ignore_index=True)

In [6]:
databases = pd.concat([
    explode_multiresponse(df, 'DatabaseHaveWorkedWith', 'Database', 'HaveWorkedWith', 'Technology'),
    explode_multiresponse(df, 'DatabaseWantToWorkWith', 'Database', 'WantToWorkWith', 'Technology'),
    explode_multiresponse(df, 'DatabaseAdmired', 'Database', 'Admired', 'Technology')
], ignore_index = True)

In [7]:
platforms = pd.concat([
    explode_multiresponse(df, 'PlatformHaveWorkedWith', 'Platform', 'HaveWorkedWith', 'Technology'),
    explode_multiresponse(df, 'PlatformWantToWorkWith', 'Platform', 'WantToWorkWith', 'Technology'),
    explode_multiresponse(df, 'PlatformAdmired', 'Platform', 'Admired', 'Technology')
], ignore_index = True)

In [8]:
collab_tools = pd.concat([
    explode_multiresponse(df, 'NEWCollabToolsHaveWorkedWith', 'Dev Tools', 'HaveWorkedWith', 'Technology'),
    explode_multiresponse(df, 'NEWCollabToolsWantToWorkWith', 'Dev Tools', 'WantToWorkWith', 'Technology'),
    explode_multiresponse(df, 'NEWCollabToolsAdmired', 'Dev Tools', 'Admired', 'Technology')
], ignore_index = True)

In [9]:
df_exploded = pd.concat([languages, databases, platforms, collab_tools], ignore_index=True)

In [10]:
df_exp = df_exploded.merge(
    df[
        ['ResponseId', 'Age_Group', 'Country', 'YearsCodePro', 'EdLevel_Group']
    ],
    on = 'ResponseId',
    how = 'left'
)

In [11]:
df_exp['Value'] = 1
df_exp.head()

,ResponseId,Technology,Category,Relation,Age_Group,Country,YearsCodePro,EdLevel_Group,Value
0,2,Bash/Shell (all shells),Language,HaveWorkedWith,35-44,GBR,17.0,Undergraduate Degree,1
1,2,Go,Language,HaveWorkedWith,35-44,GBR,17.0,Undergraduate Degree,1
2,2,HTML/CSS,Language,HaveWorkedWith,35-44,GBR,17.0,Undergraduate Degree,1
3,2,Java,Language,HaveWorkedWith,35-44,GBR,17.0,Undergraduate Degree,1
4,2,JavaScript,Language,HaveWorkedWith,35-44,GBR,17.0,Undergraduate Degree,1


In [12]:
df_exp['ResponseId'].nunique()

60059

In [13]:
df_exp.duplicated(subset=["ResponseId","Technology","Category","Relation"]).sum()

np.int64(0)

In [ ]:
df_exp["Technology"] = df_exp["Technology"].astype("category")
df_exp["Country"] = df_exp["Country"].astype("category")
df_exp["Category"] = df_exp["Category"].astype("category")
df_exp["Relation"] = df_exp["Relation"].astype("category")

In [ ]:
duplicates = df_exp.duplicated(
    subset=["ResponseId","Technology","Category","Relation"]
).sum()

print(duplicates)

In [ ]:
df_exp.shape

In [ ]:
df_exp["Value"].value_counts()

In [ ]:
df_exp["Relation"].value_counts()

In [ ]:
df_exp["Technology"].isna().sum()

In [ ]:
(df_exp["Technology"] == "").sum()

In [ ]:
print(df_exploded.shape)
print(df_exp.shape)

In [ ]:
print("Filas:", df_exp.shape[0])
print("Usuarios:", df_exp["ResponseId"].nunique())
print("Tecnologías:", df_exp["Technology"].nunique())
print("Duplicados:", df_exp.duplicated(
    subset=["ResponseId","Technology","Category","Relation"]
).sum())

In [ ]:
# Validaciones del dataset final

assert df_exp.duplicated(
    subset=["ResponseId","Technology","Category","Relation"]
).sum() == 0, "Duplicados encontrados"

assert df_exp["Value"].eq(1).all(), "Value incorrecto"

assert df_exp["Technology"].notna().all(), "Tecnologías vacías"

valid_relations = {"HaveWorkedWith","WantToWorkWith","Admired"}
assert set(df_exp["Relation"].unique()).issubset(valid_relations), \
"Relaciones inválidas"

In [ ]:
cols = [
"ResponseId",
"Technology",
"Category",
"Relation",
"Value",
"Age_Group",
"Country",
"YearsCodePro",
"EdLevel_Group",
]

df_exp = df_exp[cols]
df_exp.head()

In [ ]:
df_exp.to_csv(
    "looker_dataset.csv",
    index=False
)

In [ ]:
df_exp.shape